# **Análisis Exploratorio y Justificación Metodológica del Dataset BigFlow-NIDS**

Autor: Daniel Gomollón Embid

Trabajo Fin de Grado:

Fecha: 02/03/2026

**Universidad de Zaragoza**


----------------

Este cuaderno documenta las decisiones de ingeniería de datos y la estrategia de evaluación diseñadas para el entrenamiento del Sistema de Detección de Intrusiones (IDS).

El procesamiento de tráfico de red masivo (Big Data) para ciberseguridad presenta desafíos que para entrenar modelos basados en Machine Learning y Deep Learning.

Entre ellos destacan el *Data Leakage* temporal y la Falacia de la Tasa Base (Base Rate Fallacy). A lo largo de este documento ipynb, se visualizan y justifican matemáticamente los ajustes y técnicas aplicadas durante la fase ETL (Extract, Transform, Load) para garantizar que los modelos posteriores (Tabular ResNet + VAE) y (XGBoost + LightGBM) sean estadísticamente robustos y viables en entornos de producción reales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import os
import sys

# acceso a src/ desde /notebooks
sys.path.insert(0, os.path.join(".."))
from src.config import Config

# configuración de estilo académico para las gráficas
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# ruta de losarchivos procesados (como estoy dentro de /notebooks, subo un nivel para acceder a /data)
PROCESSED_PATH  = os.path.join("..", "data", "processed", Config.EXPERIMENT_SAVE)
RAW_PATH        = os.path.join("..", "data", "raw", Config.DATASET)
LOGS_PATH       = os.path.join("..", "outputs", "logs")
os.makedirs(LOGS_PATH, exist_ok=True)

# diccionario de mapeo de macro-clases
MACRO_CLASSES  = {i: name for i, name in enumerate(Config.CLASS_NAMES)}

# carga de los vectores objetivo (etiquetas) generados por el pipeline
print("[-] Cargando datos procesados...")
try:
    X_train = np.load(os.path.join(PROCESSED_PATH, "X_train.npy"))
    X_val   = np.load(os.path.join(PROCESSED_PATH, "X_val.npy"))
    X_test  = np.load(os.path.join(PROCESSED_PATH, "X_test.npy"))
    y_train = np.load(os.path.join(PROCESSED_PATH, "y_train.npy"))
    y_val   = np.load(os.path.join(PROCESSED_PATH, "y_val.npy"))
    y_test  = np.load(os.path.join(PROCESSED_PATH, "y_test.npy"))

    print(f"   Train : {X_train.shape} | mean={X_train.mean():.3f} | std={X_train.std():.3f}")
    print(f"   Val   : {X_val.shape}   | mean={X_val.mean():.3f} | std={X_val.std():.3f}")
    print(f"   Test  : {X_test.shape}  | mean={X_test.mean():.3f} | std={X_test.std():.3f}")

except FileNotFoundError as e:
    print(f"[!] Error: {e}")

# Alias para compatibilidad con funciones del notebook
X_train_scaled = X_train
X_val_scaled   = X_val
X_test_scaled  = X_test

### Código para determinar si hay errores en el diagnóstico de post-escalado con QuantileTransformer. La media debería estar cerca de 0 y std cerca de 1. Cusará explosión de Validation Loss

- Significa que muchas features tienen valores 0 masivamente (cuando el 80% de los valores de una feature son cero por ejemplo en muestras como Ransomware que hay pocos flujos).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_train = np.load(f"../data/processed/resultados_1_full/X_train.npy")
feature_names = np.load("../outputs/models/feature_names.npy")

# % de ceros por feature
zero_pct = (X_train == X_train.min(axis=0)).mean(axis=0) * 100

# Features con más del 50% de valores en el mínimo (ceros originales)
problematic = [(feature_names[i], zero_pct[i]) 
               for i in np.argsort(zero_pct)[::-1] 
               if zero_pct[i] > 50]

print(f"Features problemáticas (>50% en valor mínimo):")
for name, pct in problematic:
    print(f"  {name:<40} {pct:.1f}%")

print(f"\nMedia global del scaler: {X_train.mean():.3f}")
print(f"Std global del scaler:   {X_train.std():.3f}")

-----------------

## **1. El Problema Original: Distribución de Pareto y Cola Larga (Long Tail)**

Antes de aplicar cualquier técnica de balanceo, es imperativo analizar la naturaleza del dataset crudo. El tráfico de red real se caracteriza por distribuciones de Pareto extremas. En el dataset BigFlow-NIDS, los ataques volumétricos orientados a la denegación de servicio (como DDoS) generan millones de flujos debido a la propia mecánica de inundación de red.

Por el contrario, los ataques dirigidos y más sofisticados, como el *Ransomware*, las inyecciones SQL o la explotación de vulnerabilidades, generan una huella de red estadísticamente insignificante. Visualizar esta disparidad en una escala lineal estándar haría que la mayoría de las tipologías de ataque fueran invisibles.

La siguiente representación, renderizada en escala logarítmica, evidencia el desbalance estructural del conjunto de datos y justifica la necesidad metodológica de abandonar el muestreo aleatorio en favor de un enfoque estratificado.

In [ ]:
import polars as pl
import pandas as pd

print("[-] Escaneando raw parquets para calcular la distribución real...")

try:
    files = sorted(f for f in os.listdir(RAW_PATH) if f.endswith(".parquet"))
    chunks = []
    for fname in files:
        path = os.path.join(RAW_PATH, fname)
        chunk = (
            pl.scan_parquet(path)
            .with_columns([
                # fix de schema: estas columnas son Int64 en algunos parquets
                pl.col("SRC_TO_DST_SECOND_BYTES").cast(pl.Float64),
                pl.col("DST_TO_SRC_SECOND_BYTES").cast(pl.Float64),
            ])
            .select("Attack")          # solo necesitamos la columna de etiquetas
            .drop_nulls()
            .group_by("Attack")
            .agg(pl.len().alias("count"))
            .collect()                 # agrega en lazy antes de traer a RAM
        )
        chunks.append(chunk)

    raw_counts_df = (
        pl.concat(chunks)
        .group_by("Attack")
        .agg(pl.sum("count").alias("count"))   # suma conteos entre parquets
        .filter(pl.col("Attack") != "Worms")   # excluimos Worms (< 10 muestras)
        .sort("count", descending=True)
    )

    # 2. Convertimos el DataFrame a listas para Seaborn
    labels_raw  = raw_counts_df["Attack"].to_list()
    values_raw  = raw_counts_df["count"].to_list()

    # creamos DataFrame ligero para Seaborn para facilitar el mapeo de colores
    df_plot = pd.DataFrame({'Tipología': labels_raw, 'Volumen': values_raw})
    df_plot['Tipo'] = ['Benigno' if label == 'Benign' else 'Malicioso' for label in df_plot['Tipología']]

    plt.figure(figsize=(14, 7))

    # uso de hue para asignar los colores y que no salte warnings
    ax_raw = sns.barplot(
        data=df_plot, 
        x='Tipología', 
        y='Volumen', 
        hue='Tipo', 
        palette={'Benigno': '#2ecc71', 'Malicioso': '#e74c3c'},
        dodge=False # evita que las barras se hagan estrechas
    )

    plt.title('Distribución Cruda de Tipologías de Tráfico (El Problema de la Cola Larga)', fontweight='bold')
    plt.ylabel('Volumen de Flujos (Escala Logarítmica)', fontweight='bold')
    plt.xlabel('Categorías Originales del Dataset', fontweight='bold')
    plt.xticks(rotation=90, fontsize=10)

    # aplicación de escala logarítmica para que se vean bien las clases minoritarias
    ax_raw.set_yscale("log")

    # umbral de supervivencia estadístico
    plt.axhline(y=5000, color='#2980b9', linestyle='--', linewidth=2.5,
                label='Umbral de Protección Estricta (< 5.000 muestras integradas al 100%)')

    plt.legend(fontsize=11, loc='upper right', frameon=True, shadow=True)
    plt.tight_layout()

    # guardado automático para la memoria del TFG
    plt.savefig(os.path.join(LOGS_PATH, '01_long_tail_crudo_escala_log.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"[-] Operación completada. Se detectaron {len(labels_raw)} clases originales.")

except Exception as e:
    print(f"[!] Error procesando los archivos crudos: {e}")
    print("Asegúrate de que 'data/raw/BigFlow-NIDS es la ruta correcta a tus datos originales.")

--------------

## **2. Taxonomía de Ataques y Muestreo Estratificado. 8 Macro-Clases**

El dataset BigFlow-NIDS original presenta un desbalance interno extremo entre las diferentes tipologías de ataque. Mientras que los ataques volumétricos (DDoS) superan los 11 millones de registros, amenazas críticas como el *Ransomware* o los ataques *Man-in-the-Middle* (MitM) presentan menos de 5.000 muestras en total.

Un muestreo aleatorio puro habría invisibilizado estas clases críticas. Para solucionar esto, el pipeline aplica un muestreo estratificado con mínimos garantizados: se inyecta la totalidad de los registros temporales disponibles de las clases minoritarias en el conjunto de entrenamiento, y se submuestrean proporcionalmente las clases mayoritarias. Las 32 etiquetas originales se han organizado en 8 macro-clases semánticamente coherentes para facilitar la convergencia del clasificador multiclase.

In [ ]:
def plot_macro_class_distribution():
    # Aislamiento de las muestras maliciosas en el conjunto de entrenamiento
    y_train_attacks = y_train[y_train > 0]

    unique_classes, counts = np.unique(y_train_attacks, return_counts=True)

    # ordenamiento descendente para mejor visualización
    sorted_indices = np.argsort(counts)[::-1]
    sorted_counts = counts[sorted_indices]
    sorted_labels = [MACRO_CLASSES[unique_classes[i]] for i in sorted_indices]

    plt.figure(figsize=(12, 7))
    ax = sns.barplot(x=sorted_labels, y=sorted_counts, palette="mako")

    plt.title('Representatividad de Macro-Clases de Ataque en Entrenamiento', fontweight='bold')
    plt.ylabel('Frecuencia Absoluta (Número de Flujos)')
    plt.xlabel('Categoría Taxonómica (Macro-Clase)')
    plt.xticks(rotation=45, ha='right')

    # anotación de los valores exactos sobre las barras
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{int(height):,}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontweight='bold', xytext=(0, 5), textcoords='offset points')

    # contexto metodológico integrado en el gráfico
    nota_metodologica = (
        "Nota: La asimetría visible es intencional. Las clases minoritarias (ej. Malware / Botnet)\n"
        "integran el 100% de las muestras crudas disponibles para evitar la ceguera del modelo frente\n"
        "a vectores de ataque de baja frecuencia pero alto impacto."
    )
    plt.figtext(0.5, -0.25, nota_metodologica, ha="center", fontsize=11,
                bbox={"facecolor":"#f0f0f0", "alpha":0.8, "pad":8, "edgecolor":"#cccccc"})

    plt.tight_layout()
    plt.savefig('outputs/logs/01_macro_clases_train.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_macro_class_distribution()

-----------------

## **3. Mitigación de la Falacia de la Tasa Base (Prior Shift)**

Uno de los errores metodológicos más comunes en la literatura de detección de intrusos es evaluar los modelos sobre una distribución de clases artificialmente balanceada. Si un modelo se evalúa en un entorno con un 50% de ataques, las métricas globales (como la exactitud o *Accuracy*) no reflejarán su comportamiento en una red corporativa real, donde el tráfico malicioso raramente supera el 5%.

Para resolver este problema, nuestro pipeline implementa una estrategia de tres capas con distribuciones forzadas:

1. **Entrenamiento (Train - 60/40):** Sobremuestreo relativo de ataques para garantizar que la red neuronal tenga suficientes ejemplos para aprender las fronteras de decisión complejas.
2. **Validación (Val - 75/25):** Distribución intermedia para calibrar el Early Stopping y fijar el umbral de anomalía del modelo generativo (VAE) penalizando moderadamente los falsos positivos.
3. **Prueba (Test - 95/5):** Simulación estricta de un entorno de producción. Las métricas extraídas de este conjunto serán los indicadores definitivos de la viabilidad industrial del IDS.


A continuación, se visualiza la proporción de flujos benignos frente a maliciosos en cada subconjunto.

In [ ]:
def plot_distribution_shift():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Desplazamiento del Prior (Prior Shift) por Fase Experimental', fontweight='bold', y=1.05)

    datasets = [y_train, y_val, y_test]
    titles = [
        'Conjunto de Entrenamiento\n(Maximización de Aprendizaje)',
        'Conjunto de Validación\n(Calibración de Umbrales)',
        'Conjunto de Prueba\n(Simulación de Entornos Reales en Industria)'
    ]
    colors = ['#2ecc71', '#e74c3c']

    for i, ax in enumerate(axes):
        benign_count = np.sum(datasets[i] == 0)
        attack_count = np.sum(datasets[i] != 0)

        wedges, texts, autotexts = ax.pie(
            [benign_count, attack_count],
            labels=['Tráfico Benigno', 'Tráfico Malicioso'],
            autopct='%1.1f%%',
            startangle=90,
            colors=colors,
            explode=(0, 0.1),
            shadow=True
        )

        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_weight('bold')

        ax.set_title(titles[i], fontweight='bold')

    plt.tight_layout()
    os.makedirs('outputs/logs', exist_ok=True) # Crea la carpeta si no existe
    plt.savefig('outputs/logs/01_prior_shift_distribucion.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_distribution_shift()

Para alinear la salida probabilística del modelo entrenado en la distribución de entrenamiento con la realidad de la distribución de prueba, durante la fase de inferencia se aplicará una corrección bayesiana de la probabilidad a posteriori. La transformación se define mediante la siguiente ecuación:

$$P_{corregida}(\text{ataque} | x) = \frac{P_{train}(\text{ataque} | x) \cdot \frac{\pi_{prod}}{\pi_{train}}}{P_{train}(\text{ataque} | x) \cdot \frac{\pi_{prod}}{\pi_{train}} + P_{train}(\text{benigno} | x) \cdot \frac{1 - \pi_{prod}}{1 - \pi_{train}}}$$

Donde $\pi_{train}$ es la tasa base de ataques en entrenamiento (0.40) y $\pi_{prod}$ es la tasa base esperada en producción (0.05).

------------------

## **4. Ingeniería de Características (Feature Engineering)**

### 4.1. Prevención de Fugas de Información y Dispersión Estructural

Para garantizar la capacidad de generalización del modelo en entornos de producción y estabilizar el aprendizaje de la red neuronal, el dataset fue rigurosamente purgado en dos frentes: metadatos no generalizables (que provocan *Data Leakage*) y características con dispersión extrema (que envenenan el espacio latente y el escalado dimensional).

| Característica Eliminada | Motivo de Eliminación | Justificación Metodológica |
| :--- | :--- | :--- |
| **IPV4_SRC_ADDR / IPV4_DST_ADDR** | Fuga Topológica (*Leakage*) | Previene que el modelo asocie vulnerabilidades a direcciones IP específicas. El IDS debe inferir comportamientos y anomalías cinéticas, no memorizar identidades del laboratorio. |
| **L4_SRC_PORT** | Ruido Estadístico (*Leakage*) | El puerto de origen es asignado dinámicamente de forma efímera por el sistema operativo del atacante, careciendo de valor predictivo. *(Se conserva el puerto destino por identificar el servicio vulnerado).* |
| **FLOW_START_MS / FLOW_END_MS** | Fuga Temporal (*Leakage*) | Inhibe la memorización cronológica del ataque. El comportamiento del flujo se captura mediante variables invariantes en el tiempo, como la duración total y los *Inter-Arrival Times* (IAT). |
| **DNS_QUERY_ID** | Identificador Sesional (*Leakage*) | Evita el sobreajuste (*overfitting*) a números de transacción aleatorios generados en las capas de aplicación. |
| **FTP_COMMAND_RET_CODE, ICMP_TYPE, ICMP_IPV4_TYPE, DNS_TTL_ANSWER, DNS_QUERY_TYPE** | Dispersión Estructural (*Sparsity*) | Variables exclusivas de protocolos minoritarios que presentan un >80-99% de ceros estructurales. Al carecer de densidad, distorsionan severamente las transformaciones no lineales sin aportar poder discriminativo global. |
| **RETRANSMITTED_OUT_BYTES / PKTS, RETRANSMITTED_IN_BYTES / PKTS** | Dispersión Extrema (*Sparsity*) | Características presentes en menos del 6% de los flujos. Su inclusión agrava la "Maldición de la Dimensionalidad" y desestabiliza el flujo de gradientes del clasificador, siendo preferible su poda. |

### 4.2. Transformación No Lineal del Espacio Latente (Feature Scaling)

Las métricas volumétricas y temporales en ciberseguridad sufren de valores atípicos severos (outliers). Los escaladores convencionales comprimen el tráfico legítimo en rangos infinitesimales.

Mediante el uso de un Transformador Cuantílico (`QuantileTransformer`), se fuerza el mapeo de todas las características a una distribución normal estándar. Esto garantiza estabilidad numérica en el cómputo de gradientes de la red ResNet y asegura el cumplimiento matemático de la divergencia KL para la posterior integración del Autoencoder Variacional (VAE).

In [ ]:
def plot_quantile_transformation():
    try:
        sample_feature = X_train_scaled[:50000, 0]
        plt.figure(figsize=(10, 8))

        # KDE empírica pura — sin gaussiana superpuesta
        sns.histplot(sample_feature, bins=80, kde=True, color='#3498db',
                     stat="density", edgecolor="black", linewidth=0.5, alpha=0.6,
                     label='Distribución Empírica Post-QuantileTransformer (KDE)')

        plt.title('Distribución Post-QuantileTransformer — Multimodalidad por Ceros Estructurales',
                  fontweight='bold')
        plt.xlabel('Valor Escalado (Z-Score Cuantílico)', fontweight='bold')
        plt.ylabel('Densidad de Probabilidad', fontweight='bold')

        # anotación del pico estructural
        plt.axvline(x=-2, color='#e74c3c', linestyle='--', linewidth=1.8,
                    label='Cluster de ceros estructurales (x ≈ −2)')

        plt.legend(loc='upper right', frameon=True, shadow=True)

        nota_math = (
            "Interpretación:\n"
            "La multimodalidad es esperada y no problemática.\n\n"
            "— Pico en x≈−2: features de protocolo ausentes\n"
            "  (DNS, ICMP, FTP) comprimidas por el scaler.\n"
            "  SwiGLU las interpreta como 'feature apagada'.\n\n"
            "— Distribución derecha: features activas con\n"
            "  variabilidad real entre flujos benignos y ataques.\n\n"
            "— El VAE opera sobre z (128 dims), no sobre\n"
            "  estos datos crudos. Sus asunciones gaussianas\n"
            "  se satisfacen en el espacio latente, no aquí."
        )
        plt.figtext(0.62, 0.55, nota_math, fontsize=9.5,
                    bbox={"facecolor": "#f8f9fa", "alpha": 0.92, "pad": 6,
                          "edgecolor": "#cccccc"})

        plt.tight_layout()
        plt.savefig(os.path.join(LOGS_PATH, '04_distribucion_features_multimodal.png'),
                    dpi=300, bbox_inches='tight')
        plt.show()

    except Exception as e:
        print(f"[!] Error al generar gráfico de normalización: {e}")

plot_quantile_transformation()

## VER CABECERAS DE FLUJO PARA FASE DE EXPERIMENTACIÓN

In [ ]:
import polars as pl
import os

sample_path = os.path.join("../data/raw/BigFlow-NIDS", 
                            sorted(os.listdir("../data/raw/BigFlow-NIDS"))[0])

df = pl.read_parquet(sample_path)

# primeros 10 flujos de Web/Injection específicamente
web_flows = df.filter(
    pl.col("Attack").is_in(["SQL_Injection", "XSS", "Brute_Force_-Web", 
                             "Brute_Force_-XSS", "Web_Attack"])
).head(10)

print(f"Columnas disponibles ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

print(f"\nPrimeros flujos Web/Injection:")
print(web_flows.to_pandas().T)  # transpuesto para ver todos los valores

In [ ]:
# CLIENT_TCP_FLAGS=2 (SYN) con SERVER_TCP_FLAGS=0 es la firma exacta de un stealth scan
recon = df.filter(pl.col("Attack").str.contains("(?i)(scan|recon)"))
print(recon.select(["CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS", "TCP_FLAGS"]).describe())
print(recon.filter(
    (pl.col("CLIENT_TCP_FLAGS") == 2) & (pl.col("SERVER_TCP_FLAGS") == 0)
).height / recon.height)

# comprobación benignos para confirmar separación
benign = df.filter(pl.col("Attack") == "Benign")
ratio_benign = benign.filter(
    (pl.col("CLIENT_TCP_FLAGS") == 2) & (pl.col("SERVER_TCP_FLAGS") == 0)
).height / benign.height
print(f"Benign SYN_NO_RESP ratio: {ratio_benign:.4f}")

In [ ]:
import polars as pl
import os

raw_path = "../data/raw/BigFlow-NIDS"
files = sorted(f for f in os.listdir(raw_path) if f.endswith(".parquet"))

# buscar en qué parquets hay Web/Injection
web_labels = ["SQL_Injection", "XSS", "Brute_Force_-Web", 
              "Brute_Force_-XSS", "Web_Attack", "Http_Injection",
              "Web", "Injection"]

print("Buscando flujos Web/Injection en todos los parquets...")
found = []
for fname in files:
    path = os.path.join(raw_path, fname)
    attacks = pl.scan_parquet(path).select("Attack").collect()["Attack"].unique().to_list()
    web_present = [a for a in attacks if any(w.lower() in a.lower() 
                   for w in ["web", "sql", "xss", "inject", "brute"])]
    if web_present:
        print(f"  {fname}: {web_present}")
        found.append((fname, web_present))

print(f"\nTotal parquets con Web/Injection: {len(found)}")

In [ ]:
recon_tcp = (
    pl.scan_parquet(f"{DATA_PATH}/*.parquet", missing_columns='insert')
    .filter(pl.col("Attack").str.contains("(?i)(scan|recon)"))
    .select(["Attack", "TCP_FLAGS"])
    .collect()
)

benign_tcp = (
    pl.scan_parquet(f"{DATA_PATH}/*.parquet", missing_columns='insert')
    .filter(pl.col("Attack") == "Benign")
    .select(["Attack", "TCP_FLAGS"])
    .limit(100_000)
    .collect()
)

# flags con datos = TCP_FLAGS >= 8 (PSH bit activo) o flag 24 (ACK+PSH)
recon_complete  = (recon_tcp['TCP_FLAGS'] >= 8).mean()
benign_complete = (benign_tcp['TCP_FLAGS'] >= 8).mean()

print(f"Handshake completo Recon:  {recon_complete:.4f}")
print(f"Handshake completo Benign: {benign_complete:.4f}")

In [ ]:
# ver distribución completa de TCP_FLAGS en Recon
print("Recon TCP_FLAGS más frecuentes:")
print(recon_tcp['TCP_FLAGS'].value_counts().sort("count", descending=True).head(10))

print("\nBenign TCP_FLAGS más frecuentes:")
print(benign_tcp['TCP_FLAGS'].value_counts().sort("count", descending=True).head(10))

In [ ]:
recon_22  = (recon_tcp['TCP_FLAGS'] == 22).mean()
benign_22 = (benign_tcp['TCP_FLAGS'] == 22).mean()
print(f"Flag 22 (SYN+ACK+RST) Recon:  {recon_22:.4f}")
print(f"Flag 22 (SYN+ACK+RST) Benign: {benign_22:.4f}")

In [ ]:
# ¿Cuántas IPs Recon tienen >50% de sus flujos con flag 22?
# vs cuántas IPs Benign tienen >50%

recon_by_ip = (
    pl.scan_parquet(f"{DATA_PATH}/*.parquet", missing_columns='insert')
    .filter(pl.col("Attack").str.contains("(?i)(scan|recon)"))
    .select(["IPV4_SRC_ADDR", "TCP_FLAGS"])
    .collect()
    .group_by("IPV4_SRC_ADDR")
    .agg([
        (pl.col("TCP_FLAGS") == 22).mean().alias("ratio_22"),
        pl.len().alias("n_flows")
    ])
)

benign_by_ip = (
    pl.scan_parquet(f"{DATA_PATH}/*.parquet", missing_columns='insert')
    .filter(pl.col("Attack") == "Benign")
    .select(["IPV4_SRC_ADDR", "TCP_FLAGS"])
    .limit(500_000)
    .collect()
    .group_by("IPV4_SRC_ADDR")
    .agg([
        (pl.col("TCP_FLAGS") == 22).mean().alias("ratio_22"),
        pl.len().alias("n_flows")
    ])
)

print(f"IPs Recon  con ratio_22 > 0.40: {(recon_by_ip['ratio_22'] > 0.40).mean():.4f}")
print(f"IPs Benign con ratio_22 > 0.40: {(benign_by_ip['ratio_22'] > 0.40).mean():.4f}")
print(f"\nRecon  ratio_22 medio por IP: {recon_by_ip['ratio_22'].mean():.4f}")
print(f"Benign ratio_22 medio por IP: {benign_by_ip['ratio_22'].mean():.4f}")

In [ ]:
import numpy as np

# ver índices de las features del buffer
# las 13 buffer features empiezan después de las 38 base + 10 flow-level = índice 48
X = np.load("../data/processed/resultados_2_buffer/X_train.npy")
y = np.load("../data/processed/resultados_2_buffer/y_train.npy")

recon_mask  = y == 5
benign_mask = y == 0

# BUF_PORT_STD es índice 54 (48+6), BUF_PORT_RANGE es 55 (48+7)
for idx, name in [(54, "BUF_PORT_STD"), (55, "BUF_PORT_RANGE")]:
    print(f"{name}:")
    print(f"  Recon  mean={X[recon_mask,  idx].mean():.4f}")
    print(f"  Benign mean={X[benign_mask, idx].mean():.4f}")

In [ ]:
# en el parquet donde aparezcan, comparar distribución de puertos
if found:
    fname, _ = found[0]
    path = os.path.join(raw_path, fname)
    df = pl.read_parquet(path)
    
    web_flows = df.filter(
        pl.col("Attack").str.contains("(?i)(web|sql|xss|inject|brute_force)")
    )
    benign_flows = df.filter(pl.col("Attack") == "Benign").sample(n=min(100, df.height), seed=42)
    
    print(f"Web/Injection encontrados: {web_flows.height}")
    print(f"\nDistribución L4_DST_PORT (Web/Injection):")
    print(web_flows["L4_DST_PORT"].value_counts().sort("count", descending=True).head(15))
    
    print(f"\nDistribución L7_PROTO (Web/Injection):")
    print(web_flows["L7_PROTO"].value_counts().sort("count", descending=True).head(10))
    
    print(f"\nDistribución L7_PROTO (Benign):")
    print(benign_flows["L7_PROTO"].value_counts().sort("count", descending=True).head(10))
    
    print(f"\nEstadísticas Web/Injection vs Benign:")
    cols = ["IN_BYTES", "OUT_BYTES", "IN_PKTS", "OUT_PKTS", 
            "FLOW_DURATION_MILLISECONDS", "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT",
            "NUM_PKTS_UP_TO_128_BYTES", "NUM_PKTS_512_TO_1024_BYTES",
            "NUM_PKTS_1024_TO_1514_BYTES", "SRC_TO_DST_IAT_AVG", "DST_TO_SRC_IAT_AVG"]
    
    print("\nWeb/Injection — medias:")
    print(web_flows.select(cols).mean().to_pandas().T)
    
    print("\nBenign — medias:")
    print(benign_flows.select(cols).mean().to_pandas().T)

In [ ]:
print("=== Percentiles SRC_TO_DST_IAT_AVG ===")
print("Recon:")
print(recon['SRC_TO_DST_IAT_AVG'].describe())
print("\nBenign:")
print(benign['SRC_TO_DST_IAT_AVG'].describe())

In [ ]:
import numpy as np

PROC  = r"C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Provisional\data\preprocess\resultados_2_buffer"
BASE  = r"C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Provisional\outputs\models"

X     = np.load(f"{PROC}/X_test.npy")
y     = np.load(f"{PROC}/y_test.npy")
probs = np.load(f"{BASE}/test_probs.npy")

names_61 = np.load(f"{BASE}/feature_names.npy", allow_pickle=True)
names = list(names_61) + ['BUF_NEW_IP_RATIO', 'BUF_SYN_ACK_RST_RATIO']

idx_uni    = names.index('IS_UNIDIRECTIONAL')
idx_ports  = names.index('BUF_UNIQUE_DST_PORTS')
idx_rate   = names.index('BUF_SCAN_RATE')
idx_noresp = names.index('BUF_NO_RESP_RATIO')

f1 = X[:, idx_uni] * X[:, idx_ports]
f2 = X[:, idx_rate] * X[:, idx_noresp]

print(f"SCANNER_UNI_PORTS  Recon={f1[y==5].mean():.4f}  Benign={f1[y==0].mean():.4f}")
print(f"SCAN_RATE_UNI      Recon={f2[y==5].mean():.4f}  Benign={f2[y==0].mean():.4f}")

wrong   = (y==5) & (probs.argmax(axis=1)==0)
correct = (y==5) & (probs.argmax(axis=1)==5)

print(f"\nEn los mal clasificados:")
print(f"SCANNER_UNI_PORTS  wrong={f1[wrong].mean():.4f}  correct={f1[correct].mean():.4f}")
print(f"SCAN_RATE_UNI      wrong={f2[wrong].mean():.4f}  correct={f2[correct].mean():.4f}")

In [ ]:
idx_scanner = names.index('BUF_IS_SCANNER')

f3 = X[:, idx_noresp] * X[:, idx_scanner]
print(f"BUF_UNI_SCANNER_RATIO  Recon={f3[y==5].mean():.4f}  Benign={f3[y==0].mean():.4f}")
print(f"wrong={f3[wrong].mean():.4f}  correct={f3[correct].mean():.4f}")

In [ ]:
import polars as pl
import os

PARQUET_FOLDER = '../data/raw/BigFlow-NIDS'

for fname in sorted(os.listdir(PARQUET_FOLDER)):
    if not fname.endswith('.parquet'):
        continue
    df = pl.read_parquet(os.path.join(PARQUET_FOLDER, fname))
    attacks = df['Attack'].unique().to_list()
    has_recon  = any('Recon' in str(a) or 'scan' in str(a).lower() for a in attacks)
    has_benign = 'Benign' in attacks
    if has_recon or has_benign:
        print(f"{fname}: {sorted(attacks)}")

# Filtrar solo Recon y Benign
recon  = df.filter(pl.col('Attack') == 'Reconnaissance')
benign = df.filter(pl.col('Attack') == 'Benign').sample(n=min(10000, len(recon)*10), seed=42)

print(f"Recon: {len(recon)} flujos | Benign: {len(benign)} flujos\n")

# Feature 1: Asimetría de bytes
for name, subset in [('Recon', recon), ('Benign', benign)]:
    asym = (subset['OUT_BYTES'].cast(pl.Float64) / 
            (subset['IN_BYTES'].cast(pl.Float64) + subset['OUT_BYTES'].cast(pl.Float64) + 1))
    print(f"{name} BUF_BYTE_ASYMMETRY → media={asym.mean():.3f} std={asym.std():.3f}")

print()

# Feature 2: Duración cero
for name, subset in [('Recon', recon), ('Benign', benign)]:
    zero_dur = float((subset['FLOW_DURATION_MILLISECONDS'] < 10).sum()) / len(subset)
    print(f"{name} BUF_ZERO_DURATION → ratio<10ms={zero_dur:.3f}")

print()

# Feature 3: SYN sin respuesta
for name, subset in [('Recon', recon), ('Benign', benign)]:
    syn_no_resp = float(
        ((subset['CLIENT_TCP_FLAGS'] == 2) & (subset['SERVER_TCP_FLAGS'] == 0)).sum()
    ) / len(subset)
    print(f"{name} SYN_NO_RESPONSE → ratio={syn_no_resp:.3f}")

print()

# Feature 4: IAT regularidad
for name, subset in [('Recon', recon), ('Benign', benign)]:
    iat = subset['SRC_TO_DST_IAT_STDDEV'].cast(pl.Float64)
    print(f"{name} IAT_STDDEV → media={iat.mean():.1f} mediana={iat.median():.1f}")

In [ ]:
# Simular BUF_SYN_ACK_RST_RATIO y BUF_RECON_SCORE a nivel de flujo individual
# (sin buffer temporal, para ver si la señal existe en el flujo crudo)

for name, subset in [('Recon', recon), ('Benign', benign)]:
    # BUF_SYN_ACK_RST_RATIO — TCP_FLAGS == 22
    syn_ack_rst = float((subset['TCP_FLAGS'] == 22).sum()) / len(subset)
    
    # Proxy de RECON_SCORE componente 1: SYN_ACK_RST > 0.40 (a nivel de flujo es 0 o 1)
    # A nivel de buffer sería el ratio acumulado, aquí vemos la distribución base
    tcp_flags_dist = subset['TCP_FLAGS'].value_counts().sort('count', descending=True).head(8)
    
    # Asimetría OUT vs IN pkts (proxy de no_resp)
    no_resp = float((subset['OUT_PKTS'] == 0).sum()) / len(subset)
    
    # Scan rate proxy: flujos con duración < 100ms (conexiones rápidas)
    fast_flows = float((subset['FLOW_DURATION_MILLISECONDS'] < 100).sum()) / len(subset)
    
    # IAT regularidad — la que sí funcionó
    iat = subset['SRC_TO_DST_IAT_STDDEV'].cast(pl.Float64)
    
    print(f"{'='*50}")
    print(f"{name}:")
    print(f"  TCP_FLAGS==22 (SYN+ACK+RST) ratio : {syn_ack_rst:.4f}")
    print(f"  OUT_PKTS==0 (sin respuesta)  ratio : {no_resp:.4f}")
    print(f"  Flujos rápidos <100ms        ratio : {fast_flows:.4f}")
    print(f"  IAT_STDDEV media/mediana           : {iat.mean():.1f} / {iat.median():.1f}")
    print(f"\n  Top TCP_FLAGS values:")
    print(f"  {tcp_flags_dist}\n")

## Hitos para validar y estructurar fase 2 de ataques (Big 6)

In [ ]:
import numpy as np

# Necesito saber para cada feature:
# - nombre exacto (como aparece en feature_names.npy)
# - Forward (atacante controla): IN_BYTES, IN_PKTS, L4_DST_PORT...
# - Backward (respuesta servidor): OUT_BYTES, OUT_PKTS...
# - Inmutable (no tocar): PROTOCOL, L7_PROTO, timestamps...
# - BUF_* (derivadas del buffer — caso especial)
path_models = r"C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Provisional\outputs\models"
print(np.load(f'{path_models}/feature_names.npy', allow_pickle=True))

In [ ]:
path_tensores = r"C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Provisional\data\preprocess\resultados_2_buffer"

# Necesito min/max reales por feature ANTES del QuantileTransformer
# para definir los clipping bounds físicos
X_train_raw = np.load(f'{path_tensores}/X_train.npy')
# Si solo tienes el escalado, necesito también:
import joblib
scaler_global = joblib.load(f'{path_models}/quantile_scaler_global.pkl')
scaler_benign = joblib.load(f'{path_models}/quantile_scaler_benign.pkl')
buf_indices   = np.load(f'{path_models}/buf_indices.npy')
other_indices = np.load(f'{path_models}/other_indices.npy')

In [ ]:
X_train_sc = np.load(f'{path_tensores}/X_train.npy')
X_test_sc  = np.load(f'{path_tensores}/X_test.npy')
y_test     = np.load(f'{path_tensores}/y_test.npy')


# Solo ataques — los que el modelo clasifica correctamente
# (solo tiene sentido atacar lo que el modelo detecta bien)
mask_attack   = y_test != 0
X_attacks     = X_test[mask_attack]
y_attacks     = y_test[mask_attack]
print(f"Muestras de ataque para el experimento: {X_attacks.shape}")

In [ ]:
import numpy as np

feature_names = np.load(f'{path_models}/feature_names.npy', allow_pickle=True)

# ── 1. Lista de features con índice ─────────────────────────────────────
print("=" * 60)
print("FEATURES DEL DATASET")
print("=" * 60)
for i, name in enumerate(feature_names):
    tipo = "BUF_*  " if i in buf_indices else "other  "
    print(f"  [{i:>2}] {tipo}  {name}")

# ── 2. Rangos en espacio escalado (lo que el modelo ve) ─────────────────
print("\n" + "=" * 60)
print("RANGOS EN ESPACIO ESCALADO (X_train_sc)")
print("=" * 60)
print(f"  {'Feature':<30} {'Min':>8} {'Max':>8} {'Mean':>8}")
print("-" * 60)
for i, name in enumerate(feature_names):
    col = X_train_sc[:, i]
    print(f"  {name:<30} {col.min():>8.3f} {col.max():>8.3f} {col.mean():>8.3f}")

# ── 3. Rangos en espacio original (inversa del scaler) ───────────────────
print("\n" + "=" * 60)
print("RANGOS EN ESPACIO ORIGINAL (sin escalar)")
print("=" * 60)

# Invertir scaler_global sobre other_indices
X_other_sc  = X_train_sc[:, other_indices]
X_other_raw = scaler_global.inverse_transform(X_other_sc)

# Invertir scaler_benign sobre buf_indices
X_buf_sc    = X_train_sc[:, buf_indices]
X_buf_raw   = scaler_benign.inverse_transform(X_buf_sc)

# Reconstruir array completo en espacio original
X_train_raw = np.empty_like(X_train_sc)
X_train_raw[:, other_indices] = X_other_raw
X_train_raw[:, buf_indices]   = X_buf_raw

print(f"  {'Feature':<30} {'Min':>10} {'Max':>12} {'Mean':>10}")
print("-" * 68)
for i, name in enumerate(feature_names):
    col = X_train_raw[:, i]
    print(f"  {name:<30} {col.min():>10.2f} {col.max():>12.2f} {col.mean():>10.2f}")

# ── 4. Muestras de ataque correctamente clasificadas ─────────────────────
print("\n" + "=" * 60)
print("MUESTRAS DE ATAQUE DISPONIBLES PARA EL EXPERIMENTO")
print("=" * 60)
mask_attack = y_test != 0
X_attacks   = X_test_sc[mask_attack]
y_attacks   = y_test[mask_attack]
print(f"  Total ataques en test : {mask_attack.sum():,}")
print(f"  Shape X_attacks       : {X_attacks.shape}")
print(f"  Distribución por clase: {dict(zip(*np.unique(y_attacks, return_counts=True)))}")

## **Conclusiones de la Fase de Preprocesamiento**

Las visualizaciones anteriores confirman que los tensores resultantes (`X_train.npy`, `y_train.npy`, etc.) cumplen con los requisitos de diseño estipulados para la investigación:

1. El conjunto de datos está libre de variables que inducen pérdida de información futura o topológica (se eliminaron direcciones IP, puertos de origen y marcas de tiempo absolutas).
2. La partición de los datos ha respetado la secuencia física de los archivos subyacentes, simulando el comportamiento cronológico continuo de una red y bloqueando el *Data Leakage* temporal.
3. Se han resguardado las clases minoritarias, garantizando viabilidad en la posterior evaluación métrica (F1-Score macro).
4. El espacio de características ha sido transformado hacia una distribución normal mediante un `QuantileTransformer` ajustado exclusivamente sobre tráfico benigno. Esto sienta las bases matemáticas necesarias para la futura implementación del Autoencoder Variacional (VAE) y el cálculo de la distancia de Mahalanobis en el espacio latente.

El siguiente paso es la ingesta de estos tensores en la arquitectura de red neuronal residual (Tabular ResNet) estableciendo la línea base de rendimiento.